In [ ]:
from pathlib import Path
import typing as t
import polars as pl

from src.notebooks.constants import TOMATOES
from src.notebooks.config import DatasetSpec

from src.processing.orderbook import OrderBookDataset, OrderBookDataProcessor
from src.processing.constants import ORDERBOOK_FEATURES

ROUND: t.Final[int] = 0
DAY: t.Final[int] = -1
DATASET_PATH: t.Final[Path] = DatasetSpec(round_number=ROUND, day=DAY)
PRICES_PATH: t.Final[Path] = DATASET_PATH.prices()

# Orderbook Analysis

In [9]:
raw_orderbook_data: t.Final[OrderBookDataProcessor] = OrderBookDataset(PRICES_PATH).for_product(TOMATOES)
orderbook_data: t.Final[pl.DataFrame] = (
    raw_orderbook_data
    .clean()
    .add_microstructure_features()
    .add_price_features()
    .add_depth_features()
    .build()
)
orderbook_data

timestamp,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,spread,imbalance,log_returns,future_log_returns,microprice,microprice_dev,depth
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64,f64,f64,f64,f64,f64,i64
0,4999,5,4998,15,null,null,5013,5,5014,15,null,null,5006.0,14,0.0,null,0.0001,5006.0,0.0,10
100,5000,8,4998,21,null,null,5013,8,5014,21,null,null,5006.5,13,0.0,0.0001,0.0,5006.5,0.0,16
200,5000,10,4999,20,null,null,5013,10,5015,20,null,null,5006.5,13,0.0,0.0,0.0001,5006.5,0.0,20
300,5000,9,4999,21,null,null,5014,9,5015,21,null,null,5007.0,14,0.0,0.0001,0.0,5007.0,0.0,18
400,5000,5,4999,20,null,null,5014,5,5015,20,null,null,5007.0,14,0.0,0.0,0.0,5007.0,0.0,10
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
999500,4955,5,4950,10,4948,23,4963,10,4964,23,null,null,4959.0,8,0.070423,0.000605,0.0,4957.666667,-1.333333,15
999600,4955,5,4949,6,4948,18,4963,6,4964,18,null,null,4959.0,8,0.09434,0.0,-0.000504,4958.636364,-0.363636,11
999700,4950,10,4948,16,null,null,4963,10,4964,16,null,null,4956.5,13,0.0,-0.000504,0.000202,4956.5,0.0,20


In [10]:
orderbook_features: t.Final[pl.DataFrame] = orderbook_data.select(["timestamp", *ORDERBOOK_FEATURES])
orderbook_features

timestamp,mid_price,spread,imbalance,log_returns,future_log_returns,microprice,microprice_dev,depth
i64,f64,i64,f64,f64,f64,f64,f64,i64
0,5006.0,14,0.0,null,0.0001,5006.0,0.0,10
100,5006.5,13,0.0,0.0001,0.0,5006.5,0.0,16
200,5006.5,13,0.0,0.0,0.0001,5006.5,0.0,20
300,5007.0,14,0.0,0.0001,0.0,5007.0,0.0,18
400,5007.0,14,0.0,0.0,0.0,5007.0,0.0,10
…,…,…,…,…,…,…,…,…
999500,4959.0,8,0.070423,0.000605,0.0,4957.666667,-1.333333,15
999600,4959.0,8,0.09434,0.0,-0.000504,4958.636364,-0.363636,11
999700,4956.5,13,0.0,-0.000504,0.000202,4956.5,0.0,20


In [11]:
stats: t.Final[pl.DataFrame] = orderbook_features.select([
    pl.col("mid_price").mean().alias("mid_mean"),
    pl.col("mid_price").var().alias("mid_var"),
    pl.col("mid_price").std().alias("mid_std"),
    
    pl.col("microprice").mean().alias("micro_mean"),
    pl.col("microprice").var().alias("micro_var"),
    pl.col("microprice").std().alias("micro_std"),
    
    pl.col("log_returns").mean().alias("ret_mean"),
    pl.col("log_returns").var().alias("ret_var"),
    pl.col("log_returns").std().alias("ret_std"),
])
stats

mid_mean,mid_var,mid_std,micro_mean,micro_var,micro_std,ret_mean,ret_var,ret_std
f64,f64,f64,f64,f64,f64,f64,f64,f64
4977.56775,212.532588,14.578497,4977.562109,212.248573,14.568753,-9.8375e-7,7.2164e-8,0.000269


In [12]:
corrs: t.Final[pl.DataFrame] = orderbook_features.select([
    pl.corr("imbalance", "future_log_returns").alias("imb_corr"),
    pl.corr("microprice_dev", "future_log_returns").alias("micro_corr"),
])

corrs

imb_corr,micro_corr
f64,f64
-0.580231,0.331009
